# Spatial resolution: how to split California into nodes

Wave farms will connect at **coastal** substations, and the coastal grid is weak in places. How the model
splits California into nodes decides whether those limits exist in the model at all. This notebook compares
the options and lists the decisions in front of us. **We use C as the base**; section 5 explains what it reproduces
and what it doesn't.

| Option | What it is | Status |
| --- | --- | --- |
| **A. 4 ReEDS zones** | the reference's split (`base/network_2035.nc`) | solved |
| **B. 58 counties** | overlay `configs/config.california.county.yaml` (run `california-2035-county`) | solved, with the CAISO interface limits since the fix (section 4) |
| **C. 25 nodes (base)** | coastal counties kept, inland counties grouped (`configs/county_groups_25.csv`) | solved from `california_2035.ipynb` (`base/network_2035_hybrid25.nc`) |

**This notebook never solves anything.** It reads `base/spatial/`, which `scripts/export_spatial.py` fills from the
solved runs. The logic lives in `spatial_helpers.py`.

**To refresh option C:**

1. Restart the kernel of `california_2035.ipynb` (for a clean peak-RAM reading), keep its default
   `SCENARIO` and `NETWORK`, set `FORCE = True`, close other heavy apps and run it. Time and peak RAM go to
   `outputs/25bus_base_2035/run_stats.yaml`.
2. From the repo root: `pypsa-usa/.venv/bin/python scripts/export_spatial.py`.
3. Run this notebook again.

In [ ]:
import pandas as pd
import spatial_helpers as sh

pd.set_option("display.max_colwidth", 120)

## 1. The options at a glance

In [ ]:
sh.options_table()

- **A** solves in ~2 min. Gurobi's factorisation needs ~2 GB.
- **B** is ~3× larger. Its latest solve took **~30 min of computing**: the 70 min in the table include ~40 min with the
  Mac asleep, so keep it awake (`caffeinate -i`). An earlier solve without the interface limits took ~16 min. The
  factorisation alone needs ~10 GB, so the peak is probably 13–15 GB: it runs on a 17 GB laptop, but only just. The peak
  RAM of A and B was not measured.
- **C** is ~2× A and ~0.6× B in size, and solves in ~14 min. Its gain over B is memory (6 GB factorisation vs 10 GB) and
  about half the time. Its measured peak of 5.3 GB is below that estimate because macOS peak RSS leaves out compressed
  memory, so treat it as a lower bound. Solve times vary between runs: two solves of A took 2.7 and 3.9 min.


## 2. Maps

In [ ]:
sh.fig_options();

- **A:** zone p9 alone holds 44 counties, including the whole coast from Del Norte to Santa Barbara plus
  the Bay Area. Inside a zone the grid is a copper plate: power moves freely.
- **B and C** keep every coastal county as its own node. The orange lines are the weak links between the coast
  and the rest of the grid.
- The limits shown for C are those of the built network, not the plain sums of county limits (section 7).

## 3. How each coastal county is connected

In [ ]:
sh.coastal_access()

*Node export limit* is the sum of the transfer limits out of the node that holds the county.

- In **A**, Humboldt shares a copper plate with 43 other counties and an export limit of ~9.6 GW. A wave farm
  there could reach the Bay Area with no grid limit.
- In **B and C**, Humboldt can export only **216 MW** (one interface, to Trinity) and Mendocino **117 MW**.
  These are the limits that matter for North Coast wave power.
- **Del Norte** shows 0 MW: it has no link to any other California county, only to Oregon (see section 8).
- C keeps B's weak coastal limits unchanged (Humboldt, Mendocino, Santa Cruz, Marin, Sonoma). Limits that cross a
  ReEDS zone boundary are lower than in B because they are scaled to the zone-level limits of A (section 7), for
  example San Diego (6,488 → 4,702 MW) and Santa Barbara (1,119 → 986 MW). San Mateo is merged with San Francisco.

## 4. Results and run time by option

In [ ]:
sh.results_table()

In [ ]:
sh.fig_capacity();

**The interface limit fix.** B was first solved with **all six CAISO limits skipped**: with county nodes, the
external trade buses are named after counties while the limits name ReEDS zones, so no link matched. Our fork now fixes
this. Re-solving B took annual exports from 41 to 20.8 TWh, peak exports from 20.8 to 10.5 GW (the same peak as C) and
solar from 103 to 87.6 GW. To re-solve it:

```bash
cd ~/pypsa-usa/workflow   # space-free symlink, see ASSUMPTIONS.md
~/pypsa-usa/.venv/bin/python -m snakemake \
  results/california-2035-county/western/networks/elec_scounty_c58_ec_lv1.0_REM-ERM-3h_E.nc \
  -j1 --rerun-triggers mtime --forcerun add_extra_components --configfile \
  repo_data/config/config.california.yaml ../../configs/config.california.atlite.yaml \
  ../../configs/config.california.light.yaml ../../configs/config.california.county.yaml
```

**How to read cost.** *System cost* is the project's usual metric (`metrics.csv`). It includes the annualised cost
of existing assets, including the AC grid, and that grid cost depends on how the network is split: it is
recomputed link by link (~1 G$ with 3 corridors, ~3.4 G$ with 130). This is accounting, not a real difference in
the grid, so compare *system cost without AC grid*. The *solver objective* leaves out all existing assets.

**System totals barely depend on the split.** Cost without AC grid is 15.76 (A), 15.88 (B) and 15.82 G$ (C), within
0.8%, and exports are 19.7, 20.8 and 20.4 TWh. The largest gap is the CO2 price: 50, 48 and 46 $/t, which does not move
steadily with the number of nodes. What more nodes add is the location of limits and prices (section 5), which is
what wave power needs.

**Open question:** new solar is 53,408.51 MW in A and 53,408.53 MW in C, although solar output differs (159 vs 158
TWh). B builds 52,853 MW and solar moves with the export price (55.4 GW at 50 $/MWh), so this is not a fixed cap on
total solar. A match this close is hard to call chance, and it is not explained yet.

Also note: no solved run builds floating offshore wind, and Del Norte has no demand in the model.

In [ ]:
sh.fig_congestion();

Most links that bind are small rural links in the Sierra and the northern valley. On the coast, San Luis Obispo →
Santa Barbara is at its limit 66% of the time.


## 5. The proposed 25-node grouping (option C)

In [ ]:
sh.fig_hybrid("hybrid25");

In [ ]:
sh.groups_table("hybrid25")

How it was built:

- **Coast:** one node per coastal county (the 14 counties with an offshore region in PyPSA-USA). San Francisco
  has no offshore region and is merged into San Mateo.
- **Inland:** groups follow the main corridors. The North / South San Joaquin boundary is Path 15
  (Merced–Fresno); Kern / Los Angeles is Path 26. The large solar counties (Kern, San Bernardino, Riverside,
  Imperial) keep their own node so their resource is not averaged away.
- **Trinity goes inland, not with Humboldt.** Humboldt → Trinity (216 MW) is the bottleneck; Trinity → Shasta
  can carry 381 MW. Merging Trinity into Humboldt would raise Humboldt's export limit from 216 to 381 MW.
- Every group stays inside one ReEDS zone, so results can still be reported by zone.

What grouping hides: county links that bind in the 58-county run but end up inside one node.

In [ ]:
hidden = {sh.LABELS[o]: len(sh.hidden_congestion(o)) for o in sh.OPTIONS}
print("Links at their limit >= 30% of the time that become internal:", hidden)
sh.hidden_congestion("hybrid25")

C only hides small Sierra foothill links (8–209 MW), far from the coast. A hides 18, including coastal ones.

### Does C reproduce B?

**On system totals and coastal price levels, yes. On where plants are built, only partly, and two coastal areas
differ** (reading below the tables). These runs have no wave power, so a match here doesn't guarantee a match with wave.

Coastal corridors in each run: mean use is mean flow / limit, in the direction shown. Use above 1 in the 58-county
run means C's limit is lower than what B sends through that corridor, because it was scaled to a zone-level limit.


In [ ]:
sh.corridor_use()

In [ ]:
if sh.hybrid_solved():
    display(sh.node_prices())
else:
    print("Option C is not solved yet: see the steps at the top of this notebook.")

In [ ]:
sh.node_mix()

**What C reproduces:**

- **Totals:** cost without AC grid -0.4%, exports -2%. The CO2 price is 2.2 $/t lower (45.9 vs 48.1).
- **Prices:** average node prices are within 0.32 $/MWh of B, and within 0.18 $/MWh on the coast without Del Norte. Hour by hour, coastal
  prices have a correlation of 0.95–0.96 with B and differ by 1–1.4 $/MWh on average. Del Norte sits near the
  40 $/MWh export price in both runs and sets the ~9 $/MWh spread in section 4; without it the spread is 3.0 (B) and 2.4 $/MWh (C).
- **Humboldt's export limit** is unused in both runs. Humboldt and Mendocino *import* on average, so local wave
  power would first displace imports, then meet the limit.
- **Central coast:** San Luis Obispo → Santa Barbara is at its limit 66% of the time in B and 67% in C, and
  Santa Barbara → Ventura has the same mean use (47%).

**What C does not reproduce:**

- **San Diego** generates 17.6 TWh in C and 5.5 TWh in B. C caps flows into San Diego at the zone-level 1,998 MW, while B
  sums county limits to 6,083 MW and brings in 18 TWh. So C builds 1.1 GW of gas with CCS and 2.3 GW more solar
  there. A has the same 1,998 MW limit and generates 19.9 TWh there, which points to the limit scaling (section 7), not the
  grouping. This hasn't been tested by solving C without the scaling.
- **North Bay and Mendocino:** mean use rises from 6% to 57% on Upper Sacramento Valley → Sonoma, 5% to 30% on Sonoma → Marin
  and 49% to 79% on Upper Sacramento Valley → Mendocino, which C has at its limit 35% of the time. The likely
  cause is that power moves freely inside a merged inland node, while B must first cross small county links. So C
  probably overstates imports and congestion here, which could flatter wave power on this part of the coast.
- **Monterey** generates 19.6 TWh in C and 14.7 TWh in B, with 3 GW more solar. We have not looked into why.
- **Location of new plants:** about 8% of solar capacity and 31% of gas-with-CCS capacity sit at a different node.


## 6. Deciding with the wave profiles

Not every coastal county may need its own node: one with several GW of export capacity (Los Angeles, Orange,
Monterey...) could be merged into a neighbour without hiding a limit that matters. That depends on **where**
wave farms go and **how big** the largest tested farm is. The rule below keeps a coastal county separate when
its export limit is below twice the largest wave capacity tested there, so the limit could bind.
Also look at *busiest link at limit*: a corridor already congested without wave (Santa Barbara, San Luis
Obispo) is a reason to keep the county separate anyway.

**The capacities below are placeholders (500 MW at plausible sites) until the wave profiles arrive.**

In [ ]:
# PLACEHOLDER: replace with the candidate wave sites and the largest capacity tested at each (MW)
MAX_WAVE_MW = {
    "Humboldt": 500,
    "Mendocino": 500,
    "Santa Cruz": 500,
    "Monterey": 500,
    "San Luis Obispo": 500,
    "Santa Barbara": 500,
}
sh.coastal_split_check(MAX_WAVE_MW, factor=2.0)

Reading the table: *keep own node* is the rule's verdict; the last column shows what C does. Counties
marked "can merge" or "no wave planned" could be merged into a neighbour to make C smaller.

## 7. Caveat: transfer limits do not add up

In [ ]:
sh.itl_aggregation_check()

The 4-zone model uses NARIS zone-level limits. Summing the county-level limits that cross the same zone
boundary gives 1.15× to 3× more. County interfaces cannot all reach their limit at the same time.

**What C does** (`aggregate_county_itls` in our fork's `cluster_network.py`): a corridor between two nodes gets the
sum of the county limits crossing it. Corridors crossing a ReEDS zone boundary are then scaled down, per direction,
so that the boundary total equals A's zone-level limit: the p9 → p10 total is 9,328 MW in both A and C. Inside a
zone there is no reference, so the sum is kept and may still be too generous.

Corridors made of a single county interface and inside one zone are unchanged, such as Humboldt → Trinity.

## 8. Del Norte is cut off from California

In the NARIS 2024 county data, Del Norte's only interface is with Josephine County, Oregon (221 MW out,
59 MW in). It also has no demand in the model. There is no Del Norte–Humboldt line. This matches the real grid as far as we know: Del Norte is
served by Pacific Power (PacifiCorp) from southern Oregon, and PG&E's Humboldt system ends south of it.

Consequences:

- Del Norte has the largest offshore potential on the coast (9.2 GW), but its output can only be exported to
  Oregon (at 40 $/MWh, up to 221 MW).
- Option A ignored this: Del Norte sat inside p9, so its potential counted as deliverable to California.
- PyPSA-USA can only expand **existing** corridors, so a Del Norte–Humboldt line cannot be built without
  adding a candidate link in our fork. The ReEDS cost file has one (76 km).

## 9. Options in front of us

### 9.1 Spatial resolution

| | A. 4 zones | B. 58 counties | C. 25 nodes |
| --- | --- | --- | --- |
| Coastal grid limits | no (copper plate) | yes | yes; those crossing a zone boundary are scaled down |
| Inland grid limits | 3 corridors | 130 corridors | 44 corridors, small Sierra links hidden |
| Solve on a 17 GB laptop | 2–4 min, easy | ~30 min awake, at the memory limit | ~14 min, ~6 GB factorisation |
| Status | solved | solved | solved |

Revising C once the wave sites are known (section 6) only means editing `configs/county_groups_25.csv`, setting
`clusters` to the new node count and rebuilding from `cluster_network` (seconds: county profiles are reused).

### 9.2 Transfer limits inside a ReEDS zone (option C only)

Zone-boundary corridors are scaled to NARIS; corridors inside a zone keep the plain sum of county limits (section 7).
A lower, assumed limit could be tested if they turn out to matter.

### 9.3 Transmission expansion (the `ll` wildcard)

- `lv1.0` (current): no expansion. Wave value **with today's grid**: Humboldt is capped at 216 MW.
- `lv1.x` / `lc1.x`: expansion capped in volume (MW·km) or cost, placed where it helps most.
- `lvopt`: cost-optimal expansion, no cap. Shows how much grid wave power would justify.
- Expansion is only on **existing** corridors, costed with ReEDS 500 kV distance costs. New corridors
  (e.g. Del Norte–Humboldt) need a candidate link added in our fork.
- Not modelled: permitting, routing and lead times, which are severe on the North Coast.

### 9.4 Del Norte

1. No wave at Del Norte in the base (most faithful to today's grid).
2. Wave at Del Norte, exporting only to Oregon (up to 221 MW).
3. Sensitivity scenario with a candidate Del Norte–Humboldt link plus a reinforced Humboldt export.

### 9.5 If the model gets too heavy

Coarser time steps (3 h → 4–6 h), dropping linearised unit commitment, or solving on Sherlock.

### Suggested path

1. **Decided:** C is the base for the wave study. It matches B on totals and coastal prices at about half the solve
   time and with 40% less factorisation memory. Its known gaps are listed in section 5.
2. When the wave profiles arrive, fill `MAX_WAVE_MW` (section 6) and check that every county marked "yes"
   has its own node in C.
3. Run wave scenarios with `lv1.0` (today's grid) and `lvopt` (grid reinforced where it pays off).
4. Re-run one central wave scenario on B, to check C with wave power, especially at Mendocino and Sonoma.
5. If wave sites near San Diego matter, test C without scaling the limits into San Diego.
6. Del Norte: no wave in the base, with option 3 (section 9.4) as a sensitivity.
